# Publication-Grade QIMA Study for Balanced Incomplete Block Designs

**A Constraint-Preserving Quantum-Inspired Memetic Algorithm for BIBD Construction**

**Student research initiative:** Sunawar Khan  
**Supervisory research context:** Prof. Habib Hamam

This notebook is the upgraded, evidence-bound computational artifact for CSPLib Problem 028. It replaces manually entered headline results with measured runs and addresses the main methodological concerns identified during review:

1. exact and shared objective-evaluation accounting;
2. immutable raw violations separated from adaptive selection scores;
3. equal-evaluation and equal-time comparisons;
4. configurable 30-seed publication experiments;
5. component ablations that isolate the probability-register contribution;
6. exact graph-isomorphism grouping of feasible designs;
7. a sufficient Rosenberg penalty bound and an exhaustive small-case check;
8. raw CSV export and plots generated only from measured records;
9. an optional exact MILP reference baseline.

The default profile is `QUICK`, so that the notebook can be checked within a few minutes. Set `PROFILE = "PAPER"` to execute the publication experiment. No principal result is hardcoded.

In Google Colab, all outputs are saved automatically under:

`MyDrive/Outputs/BIBD_QIMA/`

## 0. Environment and experiment profile

The entire experiment is controlled from this cell. The paper profile uses at least 30 independent seeds. All stochastic methods receive either a common objective-evaluation budget or a common wall-clock budget.

In [ ]:
from __future__ import annotations

import itertools
import json
import math
import os
import platform
import sys
import time
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import networkx as nx
except ImportError:
    nx = None

try:
    from scipy.optimize import milp, LinearConstraint, Bounds
    from scipy.sparse import lil_matrix
    SCIPY_MILP_AVAILABLE = True
except Exception:
    SCIPY_MILP_AVAILABLE = False

PROFILE = "QUICK"            # Change to "PAPER" for the full study.
MASTER_SEED = 20260724
PROJECT_NAME = "BIBD_QIMA"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/Outputs") / PROJECT_NAME
else:
    OUTPUT_DIR = Path("Outputs") / PROJECT_NAME

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if PROFILE == "QUICK":
    SEEDS = list(range(3))
    EVAL_BUDGET = 6_000
    TIME_BUDGET_S = 2.0
    INSTANCES_FOR_BENCHMARK = 3
else:
    SEEDS = list(range(30))
    EVAL_BUDGET = 80_000
    TIME_BUDGET_S = 15.0
    INSTANCES_FOR_BENCHMARK = 6

np.set_printoptions(linewidth=120)
plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.grid": True, "grid.alpha": 0.22
})

ENVIRONMENT = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "platform": platform.platform(),
    "processor": platform.processor(),
    "profile": PROFILE,
    "seed_count": len(SEEDS),
    "eval_budget": EVAL_BUDGET,
    "time_budget_s": TIME_BUDGET_S,
}
print(json.dumps(ENVIRONMENT, indent=2))

## 1. BIBD formulation and exact verification

A BIBD with parameters \((v,b,r,k,\lambda)\) is represented by a binary incidence matrix \(X\in\{0,1\}^{v\times b}\). Feasibility requires:

\[
\sum_i x_{ij}=k,\qquad
\sum_j x_{ij}=r,\qquad
\sum_j x_{ij}x_{\ell j}=\lambda \ \ (i<\ell).
\]

The block-set representation stores each column as an exact \(k\)-subset. Block size is therefore satisfied structurally. Only replication and concurrence remain in the raw objective.

In [ ]:
@dataclass(frozen=True)
class BIBDParams:
    v: int
    b: int
    r: int
    k: int
    lam: int

    def label(self):
        return f"({self.v},{self.b},{self.r},{self.k},{self.lam})"

    def validate(self):
        checks = {
            "vr=bk": self.v * self.r == self.b * self.k,
            "lambda(v-1)=r(k-1)": self.lam * (self.v - 1) == self.r * (self.k - 1),
            "Fisher b>=v": self.b >= self.v,
            "incomplete k<v": self.k < self.v,
        }
        return checks


INSTANCES = [
    BIBDParams(7, 7, 3, 3, 1),
    BIBDParams(6, 10, 5, 3, 2),
    BIBDParams(9, 12, 4, 3, 1),
    BIBDParams(7, 14, 6, 3, 2),
    BIBDParams(11, 11, 5, 5, 2),
    BIBDParams(13, 13, 4, 4, 1),
]


def blocks_to_incidence(blocks, v):
    X = np.zeros((v, len(blocks)), dtype=np.int16)
    for j, block in enumerate(blocks):
        X[list(block), j] = 1
    return X


def raw_violations(blocks, p):
    X = blocks_to_incidence(blocks, p.v)
    rep = int(np.sum((X.sum(axis=1) - p.r) ** 2))
    C = X @ X.T
    pair = int(np.sum((C[np.triu_indices(p.v, 1)] - p.lam) ** 2))
    return rep, pair


def verify_design(blocks, p):
    X = blocks_to_incidence(blocks, p.v)
    C = X @ X.T
    off = C[np.triu_indices(p.v, 1)]
    return {
        "block_size_ok": bool(np.all(X.sum(axis=0) == p.k)),
        "replication_ok": bool(np.all(X.sum(axis=1) == p.r)),
        "concurrence_ok": bool(np.all(off == p.lam)),
        "identity_ok": bool(np.array_equal(C, (p.r - p.lam) * np.eye(p.v, dtype=int)
                                                + p.lam * np.ones((p.v, p.v), dtype=int))),
        "pair_values": sorted(set(int(x) for x in off)),
    }


def feasible(blocks, p):
    rep, pair = raw_violations(blocks, p)
    return rep == 0 and pair == 0


validation_table = pd.DataFrame([
    {"instance": p.label(), **p.validate()} for p in INSTANCES
])
display(validation_table)
assert validation_table.drop(columns="instance").to_numpy().all()

## 2. Faithful QUBO construction and Rosenberg penalty

For \(y=x_1x_2\), the Rosenberg gadget is

\[
R(x_1,x_2,y)=3y+x_1x_2-2x_1y-2x_2y.
\]

It is zero exactly when \(y=x_1x_2\), and at least one otherwise. A local gadget is not sufficient by itself to establish global penalty equivalence. For a concurrence term

\[
C\left(\sum_j y_j-\lambda\right)^2,
\]

changing one auxiliary variable can change the squared term by at most

\[
C\left(2\max\{\lambda,b-\lambda\}+1\right).
\]

Thus the conservative sufficient condition used here is

\[
P>C\left(2\max\{\lambda,b-\lambda\}+1\right).
\]

The former fixed value \(P=12\) does not satisfy this bound for all tested instances. The revised builder derives \(P\) automatically unless the user provides a larger value.

In [ ]:
def sufficient_rosenberg_penalty(p, concurrence_weight=3.0, safety=1.10):
    bound = concurrence_weight * (2 * max(p.lam, p.b - p.lam) + 1)
    return safety * bound


def build_qubo(p, A=6.0, B=6.0, C=3.0, P=None):
    if P is None:
        P = sufficient_rosenberg_penalty(p, C)
    strict_bound = C * (2 * max(p.lam, p.b - p.lam) + 1)
    if not P > strict_bound:
        raise ValueError(f"P={P} must exceed the sufficient bound {strict_bound}.")

    def xidx(i, j):
        return i * p.b + j

    pairs = list(itertools.combinations(range(p.v), 2))
    n_x = p.v * p.b
    yindex, nxt = {}, n_x
    for i, ell in pairs:
        for j in range(p.b):
            yindex[(i, ell, j)] = nxt
            nxt += 1

    Q = {}
    constant = A * p.b * p.k**2 + B * p.v * p.r**2 + C * len(pairs) * p.lam**2

    def add(a, c, value):
        if a > c:
            a, c = c, a
        Q[(a, c)] = Q.get((a, c), 0.0) + value

    for j in range(p.b):
        ids = [xidx(i, j) for i in range(p.v)]
        for a in ids:
            add(a, a, A * (1 - 2 * p.k))
        for a, c in itertools.combinations(ids, 2):
            add(a, c, 2 * A)

    for i in range(p.v):
        ids = [xidx(i, j) for j in range(p.b)]
        for a in ids:
            add(a, a, B * (1 - 2 * p.r))
        for a, c in itertools.combinations(ids, 2):
            add(a, c, 2 * B)

    for i, ell in pairs:
        ids = [yindex[(i, ell, j)] for j in range(p.b)]
        for a in ids:
            add(a, a, C * (1 - 2 * p.lam))
        for a, c in itertools.combinations(ids, 2):
            add(a, c, 2 * C)

    for i, ell in pairs:
        for j in range(p.b):
            z, x1, x2 = yindex[(i, ell, j)], xidx(i, j), xidx(ell, j)
            add(z, z, 3 * P)
            add(x1, x2, P)
            add(x1, z, -2 * P)
            add(x2, z, -2 * P)

    return Q, nxt, {
        "xidx": xidx, "yindex": yindex, "n_x": n_x, "constant": constant,
        "P": P, "strict_bound": strict_bound
    }


def qubo_energy(Q, sample, constant=0.0):
    return constant + sum(value * sample[a] * sample[c] for (a, c), value in Q.items())


def encode_consistent_sample(blocks, p, maps, n_vars):
    X = blocks_to_incidence(blocks, p.v)
    sample = np.zeros(n_vars, dtype=np.int8)
    for i in range(p.v):
        for j in range(p.b):
            sample[maps["xidx"](i, j)] = X[i, j]
    for (i, ell, j), idx in maps["yindex"].items():
        sample[idx] = X[i, j] * X[ell, j]
    return sample


resource_rows = []
for p in INSTANCES:
    Q, n, maps = build_qubo(p)
    resource_rows.append({
        "instance": p.label(),
        "incidence_variables": maps["n_x"],
        "Rosenberg_auxiliaries": n - maps["n_x"],
        "logical_variables_total": n,
        "candidate_block_variables": math.comb(p.v, p.k),
        "P_used": round(maps["P"], 2),
        "P_bound": maps["strict_bound"],
    })
resource_table = pd.DataFrame(resource_rows)
display(resource_table)

### Exhaustive QUBO equivalence check on a small BIBD

The instance \((3,3,2,2,1)\) requires 18 logical variables after quadratization, so all \(2^{18}\) states can be enumerated. The global QUBO minimum must have zero original BIBD violation and consistent Rosenberg auxiliaries.

In [ ]:
def exhaustive_qubo_check():
    p = BIBDParams(3, 3, 2, 2, 1)
    Q, n, maps = build_qubo(p, A=6, B=6, C=3)
    best_energy = np.inf
    best_samples = []
    for integer in range(1 << n):
        bits = np.fromiter(((integer >> q) & 1 for q in range(n)), dtype=np.int8, count=n)
        e = qubo_energy(Q, bits, maps["constant"])
        if e < best_energy - 1e-9:
            best_energy, best_samples = e, [bits.copy()]
        elif abs(e - best_energy) < 1e-9:
            best_samples.append(bits.copy())

    checks = []
    for bits in best_samples:
        blocks = []
        for j in range(p.b):
            blocks.append(tuple(i for i in range(p.v) if bits[maps["xidx"](i, j)]))
        consistent = all(
            bits[idx] == bits[maps["xidx"](i, j)] * bits[maps["xidx"](ell, j)]
            for (i, ell, j), idx in maps["yindex"].items()
        )
        checks.append(consistent and feasible(blocks, p))
    return {
        "instance": p.label(), "variables": n, "states_checked": 1 << n,
        "minimum_energy": float(best_energy), "number_of_minimizers": len(best_samples),
        "all_minimizers_valid_and_consistent": bool(all(checks))
    }


qubo_exact_check = exhaustive_qubo_check()
print(qubo_exact_check)
assert qubo_exact_check["all_minimizers_valid_and_consistent"]

## 3. Shared evaluation accounting and stopping rules

An objective evaluation is defined identically for QIMA, GA, SA, and every ablation: one calculation of the raw replication and concurrence violations for one candidate design. Repair bookkeeping and random sampling are not silently converted into fictional evaluation counts. All algorithms stop when the common evaluation or time budget is exhausted.

In [ ]:
class BudgetExceeded(RuntimeError):
    pass


@dataclass
class EvaluationBudget:
    max_evals: int | None = None
    max_seconds: float | None = None
    start: float = 0.0
    evaluations: int = 0

    def __post_init__(self):
        self.start = time.perf_counter()

    @property
    def elapsed(self):
        return time.perf_counter() - self.start

    def exhausted(self):
        return ((self.max_evals is not None and self.evaluations >= self.max_evals) or
                (self.max_seconds is not None and self.elapsed >= self.max_seconds))

    def evaluate(self, blocks, p):
        if self.exhausted():
            raise BudgetExceeded
        self.evaluations += 1
        return raw_violations(blocks, p)


def weighted_score(raw, weights):
    return weights[0] * raw[0] + weights[1] * raw[1]


def random_block(rng, p):
    return tuple(sorted(rng.choice(p.v, size=p.k, replace=False)))


def random_design(rng, p):
    return [random_block(rng, p) for _ in range(p.b)]


def canonical_block_order(blocks):
    # This removes block-order redundancy only. It is not point-label symmetry breaking.
    return sorted(tuple(sorted(block)) for block in blocks)


def repair_replication(blocks, p, rng, passes=2):
    blocks = [list(x) for x in blocks]
    for _ in range(passes):
        counts = blocks_to_incidence(blocks, p.v).sum(axis=1)
        while counts.max() > p.r and counts.min() < p.r:
            over = np.flatnonzero(counts > p.r)
            under = np.flatnonzero(counts < p.r)
            rng.shuffle(over)
            rng.shuffle(under)
            changed = False
            for old in over:
                candidate_columns = [j for j, block in enumerate(blocks) if old in block]
                rng.shuffle(candidate_columns)
                for j in candidate_columns:
                    possible = [new for new in under if new not in blocks[j]]
                    if possible:
                        new = int(rng.choice(possible))
                        blocks[j].remove(int(old))
                        blocks[j].append(new)
                        counts[old] -= 1
                        counts[new] += 1
                        changed = True
                        break
                if changed:
                    break
            if not changed:
                break
    return canonical_block_order(blocks)


def mutate_one(blocks, p, rng):
    trial = [list(x) for x in blocks]
    j = int(rng.integers(p.b))
    out_pos = int(rng.integers(p.k))
    available = [i for i in range(p.v) if i not in trial[j]]
    trial[j][out_pos] = int(rng.choice(available))
    return canonical_block_order(trial)


def local_improve(blocks, p, rng, budget, max_trials=200, targeted=False):
    best = canonical_block_order(blocks)
    try:
        best_raw = budget.evaluate(best, p)
    except BudgetExceeded:
        return best, raw_violations(best, p)

    failures = 0
    while failures < max_trials and not budget.exhausted() and sum(best_raw) > 0:
        if targeted:
            X = blocks_to_incidence(best, p.v)
            C = X @ X.T
            over_pairs = np.argwhere(np.triu(C, 1) > p.lam)
            if len(over_pairs):
                i, ell = over_pairs[int(rng.integers(len(over_pairs)))]
                shared = [j for j in range(p.b) if X[i, j] and X[ell, j]]
                if shared:
                    j = int(rng.choice(shared))
                    old = int(rng.choice([i, ell]))
                    available = [q for q in range(p.v) if q not in best[j]]
                    candidate = [list(x) for x in best]
                    candidate[j].remove(old)
                    candidate[j].append(int(rng.choice(available)))
                    trial = canonical_block_order(candidate)
                else:
                    trial = mutate_one(best, p, rng)
            else:
                trial = mutate_one(best, p, rng)
        else:
            trial = mutate_one(best, p, rng)

        try:
            trial_raw = budget.evaluate(trial, p)
        except BudgetExceeded:
            break
        if sum(trial_raw) < sum(best_raw):
            best, best_raw, failures = trial, trial_raw, 0
        else:
            failures += 1
    return best, best_raw

## 4. Revised QIMA and matched classical algorithms

The incumbent is stored through immutable raw violations. Whenever adaptive weights change, all weighted comparisons are recomputed from raw values. Feature flags create controlled ablations without changing the surrounding code.

In [ ]:
@dataclass(frozen=True)
class QIMAConfig:
    use_rotation: bool = True
    use_adaptive_weights: bool = True
    use_targeted_search: bool = True
    use_ils: bool = True
    population: int = 24
    delta0: float = 0.06
    local_every: int = 5
    stagnation_limit: int = 20


def observe(theta, p, rng):
    probabilities = np.sin(theta) ** 2
    blocks = []
    for j in range(p.b):
        weights = probabilities[:, j] + 1e-12
        weights /= weights.sum()
        blocks.append(tuple(sorted(rng.choice(p.v, p.k, replace=False, p=weights))))
    return blocks


def rotation_update(theta, blocks, p, delta):
    X = blocks_to_incidence(blocks, p.v)
    direction = np.where(X == 1, 1.0, -1.0)
    return np.clip(theta + delta * direction, 0.02, np.pi / 2 - 0.02)


def run_qima(p, seed, max_evals=None, max_seconds=None, config=QIMAConfig()):
    rng = np.random.default_rng(seed)
    budget = EvaluationBudget(max_evals=max_evals, max_seconds=max_seconds)
    theta = np.clip(np.full((p.v, p.b), np.pi / 4) +
                    rng.normal(0, 0.05, size=(p.v, p.b)), 0.02, np.pi / 2 - 0.02)
    weights = [1.0, 1.0]
    best, best_raw = None, (10**18, 10**18)
    stagnation = 0
    history = []
    generation = 0

    try:
        while not budget.exhausted():
            candidates = []
            for _ in range(config.population):
                if budget.exhausted():
                    break
                candidate = repair_replication(observe(theta, p, rng), p, rng)
                raw = budget.evaluate(candidate, p)
                candidates.append((raw, candidate))
            if not candidates:
                break

            generation_raw, generation_best = min(
                candidates, key=lambda item: weighted_score(item[0], weights)
            )

            if generation % config.local_every == 0 and not budget.exhausted():
                generation_best, generation_raw = local_improve(
                    generation_best, p, rng, budget, max_trials=120,
                    targeted=config.use_targeted_search
                )

            if sum(generation_raw) < sum(best_raw):
                best, best_raw = generation_best, generation_raw
                stagnation = 0
            else:
                stagnation += 1

            if config.use_adaptive_weights:
                if generation_raw[0] > generation_raw[1]:
                    weights[0] = min(8.0, 1.05 * weights[0])
                elif generation_raw[1] > generation_raw[0]:
                    weights[1] = min(8.0, 1.05 * weights[1])

            if config.use_rotation and best is not None:
                delta = max(0.01, config.delta0 / math.sqrt(generation + 1))
                theta = rotation_update(theta, best, p, delta)

            if stagnation >= config.stagnation_limit:
                theta = np.clip(np.full((p.v, p.b), np.pi / 4) +
                                rng.normal(0, 0.30, size=(p.v, p.b)),
                                0.02, np.pi / 2 - 0.02)
                if config.use_targeted_search and best is not None and not budget.exhausted():
                    polished, polished_raw = local_improve(
                        best, p, rng, budget, max_trials=200, targeted=True
                    )
                    if sum(polished_raw) < sum(best_raw):
                        best, best_raw = polished, polished_raw
                stagnation = 0

            history.append({
                "evaluations": budget.evaluations, "seconds": budget.elapsed,
                "violation": int(sum(best_raw))
            })
            generation += 1
            if sum(best_raw) == 0:
                break

        if config.use_ils and best is not None:
            while not budget.exhausted() and sum(best_raw) > 0:
                kicked = best
                for _ in range(3):
                    kicked = mutate_one(kicked, p, rng)
                candidate, candidate_raw = local_improve(
                    kicked, p, rng, budget, max_trials=200,
                    targeted=config.use_targeted_search
                )
                if sum(candidate_raw) < sum(best_raw):
                    best, best_raw = candidate, candidate_raw
    except BudgetExceeded:
        pass

    if best is None:
        best = random_design(rng, p)
        best_raw = raw_violations(best, p)
    return {
        "method": "QIMA", "instance": p.label(), "seed": seed,
        "feasible": int(sum(best_raw) == 0), "rep_violation": int(best_raw[0]),
        "pair_violation": int(best_raw[1]), "violation": int(sum(best_raw)),
        "evaluations": budget.evaluations, "seconds": budget.elapsed,
        "blocks": best, "history": history, "config": asdict(config)
    }


def run_ga(p, seed, max_evals=None, max_seconds=None, population_size=40):
    rng = np.random.default_rng(seed)
    budget = EvaluationBudget(max_evals=max_evals, max_seconds=max_seconds)
    population, scored = [], []
    try:
        while len(population) < population_size and not budget.exhausted():
            x = repair_replication(random_design(rng, p), p, rng, passes=1)
            population.append(x)
            scored.append((budget.evaluate(x, p), x))
        best_raw, best = min(scored, key=lambda z: sum(z[0]))
        history = []
        while not budget.exhausted() and sum(best_raw) > 0:
            scored.sort(key=lambda z: sum(z[0]))
            children = [scored[0][1], scored[min(1, len(scored)-1)][1]]
            while len(children) < population_size and not budget.exhausted():
                def tournament():
                    ids = rng.choice(len(scored), size=min(3, len(scored)), replace=False)
                    return min((scored[i] for i in ids), key=lambda z: sum(z[0]))[1]
                a, b = tournament(), tournament()
                child = [a[j] if rng.random() < 0.5 else b[j] for j in range(p.b)]
                if rng.random() < 0.85:
                    child = mutate_one(child, p, rng)
                child = repair_replication(child, p, rng, passes=1)
                children.append(child)
            scored = []
            for child in children:
                if budget.exhausted():
                    break
                scored.append((budget.evaluate(child, p), child))
            if scored:
                candidate_raw, candidate = min(scored, key=lambda z: sum(z[0]))
                if sum(candidate_raw) < sum(best_raw):
                    best_raw, best = candidate_raw, candidate
            history.append({"evaluations": budget.evaluations, "seconds": budget.elapsed,
                            "violation": int(sum(best_raw))})
    except BudgetExceeded:
        pass
    return {
        "method": "GA", "instance": p.label(), "seed": seed,
        "feasible": int(sum(best_raw) == 0), "rep_violation": int(best_raw[0]),
        "pair_violation": int(best_raw[1]), "violation": int(sum(best_raw)),
        "evaluations": budget.evaluations, "seconds": budget.elapsed,
        "blocks": best, "history": history
    }


def run_sa(p, seed, max_evals=None, max_seconds=None, T0=4.0):
    rng = np.random.default_rng(seed)
    budget = EvaluationBudget(max_evals=max_evals, max_seconds=max_seconds)
    current = repair_replication(random_design(rng, p), p, rng, passes=1)
    current_raw = budget.evaluate(current, p)
    best, best_raw = current, current_raw
    history = []
    try:
        while not budget.exhausted() and sum(best_raw) > 0:
            progress = (budget.evaluations / max_evals) if max_evals else min(1.0, budget.elapsed / max_seconds)
            temperature = T0 * (1 - progress) + 1e-3
            trial = mutate_one(current, p, rng)
            trial_raw = budget.evaluate(trial, p)
            difference = sum(trial_raw) - sum(current_raw)
            if difference <= 0 or rng.random() < math.exp(-difference / temperature):
                current, current_raw = trial, trial_raw
                if sum(current_raw) < sum(best_raw):
                    best, best_raw = current, current_raw
            if budget.evaluations % 250 == 0:
                history.append({"evaluations": budget.evaluations, "seconds": budget.elapsed,
                                "violation": int(sum(best_raw))})
    except BudgetExceeded:
        pass
    return {
        "method": "SA", "instance": p.label(), "seed": seed,
        "feasible": int(sum(best_raw) == 0), "rep_violation": int(best_raw[0]),
        "pair_violation": int(best_raw[1]), "violation": int(sum(best_raw)),
        "evaluations": budget.evaluations, "seconds": budget.elapsed,
        "blocks": best, "history": history
    }

## 5. Unit and invariance checks

These tests protect the claims made by the notebook. They verify feasibility, exact counter behavior, budget compliance, representation validity, and deterministic replay under a fixed seed.

In [ ]:
def run_unit_tests():
    p = INSTANCES[0]
    fano = [(0,1,5), (0,2,4), (0,3,6), (1,2,6), (1,3,4), (2,3,5), (4,5,6)]
    assert feasible(fano, p)
    assert verify_design(fano, p)["identity_ok"]

    counter = EvaluationBudget(max_evals=2)
    counter.evaluate(fano, p)
    counter.evaluate(fano, p)
    assert counter.evaluations == 2 and counter.exhausted()

    r1 = run_qima(p, seed=91, max_evals=500)
    r2 = run_qima(p, seed=91, max_evals=500)
    assert r1["blocks"] == r2["blocks"]
    assert r1["evaluations"] <= 500 and r2["evaluations"] <= 500
    assert all(len(block) == p.k and len(set(block)) == p.k for block in r1["blocks"])

    for runner in (run_qima, run_ga, run_sa):
        result = runner(p, seed=3, max_evals=250)
        assert result["evaluations"] <= 250
        assert result["violation"] == result["rep_violation"] + result["pair_violation"]
    return "All unit tests passed."


print(run_unit_tests())

## 6. Measured benchmark runner

Two protocols are executed independently:

- **Equal evaluations:** every method receives the same objective-evaluation budget.
- **Equal time:** every method receives the same wall-clock budget.

Every run is saved as a raw row. Summary tables are derived after the raw files are written.

In [ ]:
def serializable_record(result, protocol):
    return {
        "protocol": protocol, "method": result["method"], "instance": result["instance"],
        "seed": result["seed"], "feasible": result["feasible"],
        "rep_violation": result["rep_violation"], "pair_violation": result["pair_violation"],
        "violation": result["violation"], "evaluations": result["evaluations"],
        "seconds": result["seconds"],
        "blocks_json": json.dumps([list(map(int, b)) for b in result["blocks"]])
    }


def run_benchmark(instances, seeds, protocol):
    rows, histories = [], []
    runners = [run_qima, run_ga, run_sa]
    for p in instances:
        for seed in seeds:
            for runner in runners:
                kwargs = {"max_evals": EVAL_BUDGET} if protocol == "equal_evaluations" else {"max_seconds": TIME_BUDGET_S}
                result = runner(p, seed=seed, **kwargs)
                rows.append(serializable_record(result, protocol))
                for point in result["history"]:
                    histories.append({
                        "protocol": protocol, "method": result["method"],
                        "instance": p.label(), "seed": seed, **point
                    })
                print(protocol, p.label(), seed, result["method"],
                      "OK" if result["feasible"] else f"v={result['violation']}",
                      f"evals={result['evaluations']}", f"time={result['seconds']:.3f}s")
    return pd.DataFrame(rows), pd.DataFrame(histories)


benchmark_instances = INSTANCES[:INSTANCES_FOR_BENCHMARK]
eval_results, eval_history = run_benchmark(benchmark_instances, SEEDS, "equal_evaluations")
time_results, time_history = run_benchmark(benchmark_instances, SEEDS, "equal_time")

raw_results = pd.concat([eval_results, time_results], ignore_index=True)
raw_history = pd.concat([eval_history, time_history], ignore_index=True)
raw_results.to_csv(OUTPUT_DIR / f"raw_results_{PROFILE.lower()}.csv", index=False)
raw_history.to_csv(OUTPUT_DIR / f"raw_history_{PROFILE.lower()}.csv", index=False)
with open(OUTPUT_DIR / f"environment_{PROFILE.lower()}.json", "w") as stream:
    json.dump(ENVIRONMENT, stream, indent=2)

print("Saved", len(raw_results), "measured run records.")
display(raw_results.head())

## 7. Statistical summary with Wilson confidence intervals

Success rates are shown with 95% Wilson intervals. Median time, median evaluations, and median residual violation are reported because small heuristic samples are rarely normally distributed.

In [ ]:
def wilson_interval(successes, n, z=1.959963984540054):
    if n == 0:
        return np.nan, np.nan
    phat = successes / n
    denominator = 1 + z*z/n
    centre = (phat + z*z/(2*n)) / denominator
    half = z * math.sqrt(phat*(1-phat)/n + z*z/(4*n*n)) / denominator
    return centre - half, centre + half


def summarize(df):
    rows = []
    for keys, group in df.groupby(["protocol", "instance", "method"], sort=False):
        successes = int(group["feasible"].sum())
        low, high = wilson_interval(successes, len(group))
        rows.append({
            "protocol": keys[0], "instance": keys[1], "method": keys[2],
            "runs": len(group), "successes": successes,
            "success_rate": successes / len(group),
            "wilson_low": low, "wilson_high": high,
            "median_violation": group["violation"].median(),
            "iqr_violation": group["violation"].quantile(.75) - group["violation"].quantile(.25),
            "median_evaluations": group["evaluations"].median(),
            "median_seconds": group["seconds"].median(),
        })
    return pd.DataFrame(rows)


summary = summarize(raw_results)
summary.to_csv(OUTPUT_DIR / f"summary_{PROFILE.lower()}.csv", index=False)
display(summary.round(4))

## 8. Component ablation

The ablation uses the same evaluation budget for every configuration. The matched classical memetic variant disables rotation and adaptive weights while retaining the same local-search and ILS machinery.

In [ ]:
ABLATIONS = {
    "Full QIMA": QIMAConfig(),
    "No rotation": QIMAConfig(use_rotation=False),
    "No adaptive weights": QIMAConfig(use_adaptive_weights=False),
    "No targeted search": QIMAConfig(use_targeted_search=False),
    "No ILS": QIMAConfig(use_ils=False),
    "Matched classical memetic": QIMAConfig(use_rotation=False, use_adaptive_weights=False),
}


def run_ablation(instances, seeds):
    rows = []
    for p in instances:
        for seed in seeds:
            for name, config in ABLATIONS.items():
                result = run_qima(p, seed=seed, max_evals=EVAL_BUDGET, config=config)
                row = serializable_record(result, "ablation_equal_evaluations")
                row["configuration"] = name
                rows.append(row)
                print(p.label(), seed, name, "OK" if result["feasible"] else result["violation"])
    return pd.DataFrame(rows)


# In QUICK mode, ablation is limited to the hardest instance currently included.
# In PAPER mode, it covers the two structurally hardest designs.
ablation_instances = ([benchmark_instances[-1]] if PROFILE == "QUICK"
                      else [INSTANCES[-2], INSTANCES[-1]])
ablation_results = run_ablation(ablation_instances, SEEDS)
ablation_results.to_csv(OUTPUT_DIR / f"ablation_raw_{PROFILE.lower()}.csv", index=False)

ablation_summary = []
for (instance, configuration), group in ablation_results.groupby(["instance", "configuration"], sort=False):
    s = int(group.feasible.sum())
    low, high = wilson_interval(s, len(group))
    ablation_summary.append({
        "instance": instance, "configuration": configuration, "runs": len(group),
        "success_rate": s/len(group), "wilson_low": low, "wilson_high": high,
        "median_violation": group.violation.median(),
        "median_seconds": group.seconds.median()
    })
ablation_summary = pd.DataFrame(ablation_summary)
display(ablation_summary.round(4))

## 9. Exact isomorphism grouping

A feasible BIBD is converted into a bipartite incidence graph. Exact labeled-graph isomorphism is then used to cluster outputs. This avoids interpreting point relabelings or block reorderings as non-isomorphic designs.

In [ ]:
def incidence_graph(blocks, p):
    if nx is None:
        raise RuntimeError("networkx is required for exact graph-isomorphism grouping.")
    G = nx.Graph()
    for i in range(p.v):
        G.add_node(("point", i), bipartite=0)
    for j in range(p.b):
        G.add_node(("block", j), bipartite=1)
        for i in blocks[j]:
            G.add_edge(("point", int(i)), ("block", j))
    return G


def exact_isomorphism_classes(block_lists, p):
    if nx is None:
        return {"available": False, "classes": None}
    node_match = nx.algorithms.isomorphism.categorical_node_match("bipartite", -1)
    representatives, classes = [], []
    for blocks in block_lists:
        graph = incidence_graph(blocks, p)
        placed = False
        for idx, representative in enumerate(representatives):
            if nx.is_isomorphic(graph, representative, node_match=node_match):
                classes[idx].append(blocks)
                placed = True
                break
        if not placed:
            representatives.append(graph)
            classes.append([blocks])
    return {"available": True, "classes": classes}


iso_rows = []
for p in benchmark_instances:
    group = eval_results[(eval_results.instance == p.label()) &
                         (eval_results.method == "QIMA") &
                         (eval_results.feasible == 1)]
    designs = [[tuple(block) for block in json.loads(s)] for s in group.blocks_json]
    result = exact_isomorphism_classes(designs, p)
    iso_rows.append({
        "instance": p.label(), "feasible_outputs": len(designs),
        "exact_isomorphism_classes": len(result["classes"]) if result["available"] else np.nan,
        "networkx_available": result["available"]
    })
isomorphism_table = pd.DataFrame(iso_rows)
display(isomorphism_table)

## 10. Optional exact MILP reference baseline

For modest instances, one binary variable is assigned to each possible \(k\)-subset. The linear constraints enforce exactly \(b\) selected blocks, replication \(r\), and pair concurrence \(\lambda\). This formulation produces simple BIBDs because each candidate block can be selected at most once. It is reported separately from the matched stochastic-budget comparison.

In [ ]:
def solve_candidate_block_milp(p, time_limit=30.0):
    if not SCIPY_MILP_AVAILABLE:
        return {"available": False, "message": "scipy.optimize.milp is unavailable."}
    candidates = list(itertools.combinations(range(p.v), p.k))
    pairs = list(itertools.combinations(range(p.v), 2))
    rows = 1 + p.v + len(pairs)
    A = lil_matrix((rows, len(candidates)), dtype=float)
    rhs = np.empty(rows)
    A[0, :] = 1
    rhs[0] = p.b
    for i in range(p.v):
        for q, block in enumerate(candidates):
            if i in block:
                A[1+i, q] = 1
        rhs[1+i] = p.r
    for idx, pair in enumerate(pairs):
        for q, block in enumerate(candidates):
            if pair[0] in block and pair[1] in block:
                A[1+p.v+idx, q] = 1
        rhs[1+p.v+idx] = p.lam
    result = milp(
        c=np.zeros(len(candidates)), integrality=np.ones(len(candidates)),
        bounds=Bounds(np.zeros(len(candidates)), np.ones(len(candidates))),
        constraints=LinearConstraint(A.tocsr(), rhs, rhs),
        options={"time_limit": time_limit}
    )
    selected = []
    if result.x is not None:
        selected = [candidates[i] for i, value in enumerate(result.x) if value > 0.5]
    return {
        "available": True, "success": bool(result.success),
        "status": int(result.status), "message": result.message,
        "candidate_variables": len(candidates), "blocks": selected,
        "verified": bool(len(selected) == p.b and feasible(selected, p))
    }


milp_demo = solve_candidate_block_milp(INSTANCES[0], time_limit=10.0)
print({k: v for k, v in milp_demo.items() if k != "blocks"})
if milp_demo.get("success"):
    assert milp_demo["verified"]

## 11. Figures generated from measured data

No values are entered manually. Error bars show Wilson uncertainty in success probability.

In [ ]:
COLORS = {"QIMA": "#167D8D", "GA": "#D49A2A", "SA": "#596A78"}

def plot_success(summary_df, protocol):
    data = summary_df[summary_df.protocol == protocol].copy()
    instances = list(dict.fromkeys(data.instance))
    methods = ["QIMA", "GA", "SA"]
    x = np.arange(len(instances))
    width = 0.25
    fig, ax = plt.subplots(figsize=(10, 4.8))
    for m_idx, method in enumerate(methods):
        subset = data[data.method == method].set_index("instance").reindex(instances)
        values = subset.success_rate.to_numpy()
        low = np.maximum(0.0, values - subset.wilson_low.to_numpy())
        high = np.maximum(0.0, subset.wilson_high.to_numpy() - values)
        ax.bar(x + (m_idx-1)*width, values, width, label=method, color=COLORS[method],
               yerr=np.vstack([low, high]), capsize=3)
    ax.set_xticks(x, instances, rotation=25, ha="right")
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("Exact-feasibility probability")
    ax.set_title(protocol.replace("_", " ").title())
    ax.legend(frameon=False)
    fig.tight_layout()
    path = OUTPUT_DIR / f"success_{protocol}_{PROFILE.lower()}.png"
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    return path


figure_eval = plot_success(summary, "equal_evaluations")
figure_time = plot_success(summary, "equal_time")

fig, ax = plt.subplots(figsize=(9.5, 4.8))
for instance in ablation_summary.instance.unique():
    subset = ablation_summary[ablation_summary.instance == instance]
    ax.plot(subset.configuration, subset.success_rate, marker="o", label=instance)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Exact-feasibility probability")
ax.set_title("QIMA component ablation under equal evaluation budgets")
ax.tick_params(axis="x", rotation=28)
ax.legend(frameon=False)
fig.tight_layout()
ablation_figure = OUTPUT_DIR / f"ablation_{PROFILE.lower()}.png"
fig.savefig(ablation_figure, bbox_inches="tight")
plt.show()

## 12. Evidence-bound interpretation checklist

Before manuscript values are updated, the `PAPER` profile should be run to completion and the following conditions should be checked:

- The raw result file contains all expected runs.
- Evaluation counts never exceed the shared budget.
- Every reported percentage is derived from raw records.
- Wilson intervals accompany success rates.
- The matched classical memetic ablation is compared with full QIMA.
- Diversity is described as exact incidence-graph isomorphism classes.
- QUBO counts are called logical-variable counts before embedding.
- Failure to solve a QUBO with a heuristic sampler is not described as failure of the encoding.
- Any statement of superiority is restricted to the tested methods, budgets, and instances.

The strongest defensible conclusion should be selected only after the full results are observed. The notebook intentionally does not contain a predetermined superiority statement.

In [ ]:
expected_runs = 2 * len(benchmark_instances) * len(SEEDS) * 3
assert len(raw_results) == expected_runs
assert (eval_results.evaluations <= EVAL_BUDGET).all()
assert set(raw_results.method) == {"QIMA", "GA", "SA"}
assert raw_results[["seconds", "evaluations", "violation"]].notna().all().all()

manifest = {
    "profile": PROFILE,
    "expected_run_records": expected_runs,
    "actual_run_records": len(raw_results),
    "raw_results": str(OUTPUT_DIR / f"raw_results_{PROFILE.lower()}.csv"),
    "raw_history": str(OUTPUT_DIR / f"raw_history_{PROFILE.lower()}.csv"),
    "summary": str(OUTPUT_DIR / f"summary_{PROFILE.lower()}.csv"),
    "ablation": str(OUTPUT_DIR / f"ablation_raw_{PROFILE.lower()}.csv"),
    "figures": [str(figure_eval), str(figure_time), str(ablation_figure)],
    "qubo_exact_check": qubo_exact_check,
}
with open(OUTPUT_DIR / f"manifest_{PROFILE.lower()}.json", "w") as stream:
    json.dump(manifest, stream, indent=2)
print(json.dumps(manifest, indent=2))
print("Notebook audit complete.")